# 04 · Function calling: the tool loop, end to end

After this notebook you can define tools the model picks correctly, run the full loop (model → tool calls → your code executes → results back → final answer), including parallel calls, errors and argument validation. You'll also know the two message-ordering bugs that cause most 400s, and how the Chat Completions and Responses APIs differ.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · OpenAI API basics](01_openai_api_basics.ipynb), [03 · Structured outputs](03_structured_outputs_with_pydantic.ipynb)

## Why this matters in interviews

- Every agent is this loop plus guards. "Walk me through what happens when the model calls a tool" is a standard screen, and "build an agent in 30 minutes" coding rounds start here.
- The security point interviewers listen for: **the model never executes anything**. It proposes a call and your code decides. Validation, authorisation and approval all live in that gap.
- Tool design questions (names, descriptions, enums, how many tools) are really prompt-engineering questions. Descriptions are prompts.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag03` | Are function calling and tool calling the same thing? |
| `ag02` | What is the core agent loop? |
| `ag04` | What makes a good tool definition? |
| `ag18` | How do you scope tools so the agent picks the right one? |
| `ag19` | How do you handle a tool that fails? |
| `pr18` | How do you pass tool definitions efficiently? |
| `po28` | Design the path between an agent proposing a tool call and the call executing. |
| `tl23` | How would you expose an existing internal REST API to an AI agent? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import ast, inspect, json, operator, re
from typing import Annotated, Literal, get_type_hints
import matplotlib.pyplot as plt
import openai
from pydantic import BaseModel, Field, ValidationError, create_model, field_validator

def api_message(e):
    return e.body.get("message", str(e)) if isinstance(e.body, dict) else str(e)

print("provider:", PROVIDER, "| model:", MODEL)

## 1. The idea in one picture

**In plain English:** you give the model a menu of functions it may *ask* for. Instead of answering, it can reply "please run `get_weather(city="Paris")`". Your code runs it (or refuses to), sends back the result, and the model writes the answer. "Function calling" is the original 2023 name and "tool calling" the current one. Same mechanism.

In [ ]:
def draw_sequence(actors, steps, title=None):
    """A minimal sequence diagram: lifelines plus labelled arrows, top to bottom."""
    n = len(steps)
    fig, ax = plt.subplots(figsize=(10.5, 0.52 * n + 1.3))
    xs = {a: i * 3.2 for i, a in enumerate(actors)}
    for a, x in xs.items():
        ax.text(x, n + 0.55, a, ha="center", va="center", fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.35", fc="#e7f0ff", ec="#3b5bdb"))
        ax.plot([x, x], [-0.4, n + 0.15], color="#adb5bd", lw=1, ls="--", zorder=0)
    for i, (src, dst, label) in enumerate(steps):
        y = n - i - 0.3
        color = "#e8590c" if "tool" in label.lower() and dst in ("Your code", "Tools") else "#1c7ed6"
        ax.annotate("", xy=(xs[dst], y), xytext=(xs[src], y), arrowprops=dict(arrowstyle="->", lw=1.5, color=color))
        ax.text((xs[src] + xs[dst]) / 2, y + 0.1, label, ha="center", va="bottom", fontsize=8)
    ax.set_xlim(-1.6, max(xs.values()) + 1.6); ax.set_ylim(-0.7, n + 1.0); ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()

draw_sequence(["User", "Your code", "Model API", "Tools"], [
    ("User", "Your code", "\"Weather in Paris and Tokyo?\""),
    ("Your code", "Model API", "messages + tool schemas"),
    ("Model API", "Your code", "finish_reason=tool_calls: 2 calls (ids, JSON args)"),
    ("Your code", "Tools", "validate, then get_weather('Paris'), get_weather('Tokyo')"),
    ("Tools", "Your code", "results (or errors, as data)"),
    ("Your code", "Model API", "messages + assistant(tool_calls) + 2 tool messages"),
    ("Model API", "Your code", "finish_reason=stop: final text"),
    ("Your code", "User", "answer"),
], title="One tool round trip: the model proposes, your code executes")

## 2. Anatomy of a tool definition

| Part | What it's for | Good | Bad |
|---|---|---|---|
| `name` | what the model "calls" | `get_order_status` (verb_noun, specific) | `data`, `helper`, `api2` |
| `description` | **this is a prompt**: when to use it, what it returns, when *not* to use it | "Look up the shipping status of an order by id. Not for refunds." | "Order tool." |
| `parameters` | JSON Schema of the arguments | enums for closed sets, a description per field | free-text `status` that invites "Open"/"open"/"OPEN" |
| `strict: true` | arguments guaranteed to match the schema | needs `additionalProperties: false` and all fields `required` | leaving it off, then parsing whatever arrives |

In [ ]:
WEATHER_TOOL = {"type": "function", "function": {
    "name": "get_weather",
    "description": "Get the current weather for one city. Call once per city.",
    "strict": True,
    "parameters": {"type": "object", "additionalProperties": False, "required": ["city", "unit"],
                   "properties": {"city": {"type": "string", "description": "City name, e.g. Paris"},
                                  "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}}}}
ORDER_TOOL = {"type": "function", "function": {
    "name": "get_order_status",
    "description": "Look up the shipping status of an order by its order id (like ORD12345). Not for refunds.",
    "strict": True,
    "parameters": {"type": "object", "additionalProperties": False, "required": ["order_id"],
                   "properties": {"order_id": {"type": "string", "description": "Order id, e.g. ORD12345"}}}}}
CALC_TOOL = {"type": "function", "function": {
    "name": "calculator",
    "description": "Evaluate an arithmetic expression exactly, e.g. 17*23 + 4. Use for any maths.",
    "strict": True,
    "parameters": {"type": "object", "additionalProperties": False, "required": ["expression"],
                   "properties": {"expression": {"type": "string"}}}}}
TOOLS = [WEATHER_TOOL, ORDER_TOOL, CALC_TOOL]

And the Python side: the real implementations (fake data here). A tool returns **data** (a dict that becomes a JSON string). On failure it returns an error *as data* rather than raising, so the model can read it and recover.

In [ ]:
FAKE_WEATHER = {"paris": 18, "tokyo": 24, "london": 12, "lima": 21, "oslo": 7}
FAKE_ORDERS = {"ORD55123": {"status": "shipped", "carrier": "DHL", "eta": "2026-10-02"}}

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg}

def safe_calc(expression):
    """Arithmetic only -- walks the syntax tree instead of calling eval()."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only arithmetic is allowed")
    return ev(ast.parse(expression, mode="eval").body)

def get_weather(city, unit="celsius"):
    c = FAKE_WEATHER.get(city.lower())
    if c is None:
        return {"error": f"no weather data for {city!r}"}
    return {"city": city, "temp": c if unit == "celsius" else round(c * 9 / 5 + 32), "unit": unit, "sky": "cloudy"}

def get_order_status(order_id):
    return FAKE_ORDERS.get(order_id, {"error": f"order {order_id} not found"})

def calculator(expression):
    try:
        return {"expression": expression, "result": safe_calc(expression)}
    except (ValueError, SyntaxError, ZeroDivisionError) as e:
        return {"error": f"cannot evaluate {expression!r}: {e}"}

REGISTRY = {"get_weather": get_weather, "get_order_status": get_order_status, "calculator": calculator}

## 3. The model asks; you look at what it asked for

In [ ]:
messages = [{"role": "user", "content": "Where is my order ORD55123?"}]
r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
msg = r.choices[0].message
print("finish_reason:", r.choices[0].finish_reason)
print("content      :", msg.content)
for tc in msg.tool_calls:
    print(f"tool call    : id={tc.id}  name={tc.function.name}  arguments={tc.function.arguments!r}")
assert r.choices[0].finish_reason == "tool_calls" and msg.tool_calls[0].function.name == "get_order_status"

Three things to notice. `content` is `None`: the model answered with a *request*, not text. `arguments` is a **JSON string**, not a dict: you parse it, and you validate it before trusting it. And every call has an `id`, which your reply must quote back.

## 4. The full loop

1. Send messages + tools.
2. If the reply has `tool_calls`: append that **assistant message (with its `tool_calls`)**, then run each call and append one `{"role": "tool", "tool_call_id": ..., "content": ...}` per call.
3. Call the model again with the longer list. Repeat until it answers with text or you hit the step cap.

In [ ]:
def run_tools(tool_calls, registry=REGISTRY):
    """Execute tool calls -> list of tool messages. Unknown tools and bad JSON become error *results*."""
    out = []
    for tc in tool_calls:
        fn = registry.get(tc.function.name)
        try:
            args = json.loads(tc.function.arguments)
            result = fn(**args) if fn else {"error": f"unknown tool {tc.function.name}"}
        except (json.JSONDecodeError, TypeError) as e:
            result = {"error": f"bad arguments: {e}"}
        out.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return out

def chat_with_tools(user_text, tools=TOOLS, max_steps=5, verbose=True, **kw):
    messages = [{"role": "system", "content": "Use the tools for facts you don't have. Be brief."},
                {"role": "user", "content": user_text}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, **kw)
        msg = r.choices[0].message
        if not msg.tool_calls:                          # plain text = the final answer
            if verbose:
                print(f"step {step}: final answer -> {msg.content}")
            return msg.content, messages
        messages.append({"role": "assistant", "content": msg.content,          # 1) the request itself...
                         "tool_calls": [{"id": tc.id, "type": "function",
                                         "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                        for tc in msg.tool_calls]})
        results = run_tools(msg.tool_calls)
        messages.extend(results)                                                 # 2) ...then one result per call
        if verbose:
            for tc, res in zip(msg.tool_calls, results):
                print(f"step {step}: {tc.function.name}({tc.function.arguments}) -> {res['content']}")
    raise RuntimeError(f"no final answer after {max_steps} steps")               # never loop forever

answer, transcript = chat_with_tools("Where is my order ORD55123?")
print("\nroles sent on the last call:", [m["role"] for m in transcript])
assert "shipped" in answer.lower()

With a real model the final answer reads like "Your order ORD55123 has shipped with DHL, ETA 2 October." Offline, the stand-in quotes the tool result verbatim ("Based on the tool results: …"). The *plumbing* is identical, and it is the part you are responsible for.

## 5. Parallel tool calls

Ask about two cities and a capable model requests **both calls in one turn**. You run them (concurrently, if they are slow) and return **one tool message per call id**, in any order.

In [ ]:
answer, transcript = chat_with_tools("What's the weather in Paris and in Tokyo right now?")
assistant_turn = next(m for m in transcript if m["role"] == "assistant")
print("\ncalls in one assistant turn:", len(assistant_turn["tool_calls"]))
assert len(assistant_turn["tool_calls"]) == 2 and "18" in answer and "24" in answer

r = client.chat.completions.create(model=MODEL, tools=TOOLS, parallel_tool_calls=False,
                                   messages=[{"role": "user", "content": "What's the weather in Paris and in Tokyo right now?"}])
print("with parallel_tool_calls=False:", len(r.choices[0].message.tool_calls), "call per turn")

`parallel_tool_calls=False` forces one call per turn. Use it when calls depend on each other or must not run concurrently (two writes to the same record). The loop above handles either case unchanged.

## 6. `tool_choice`: who decides whether a tool runs

In [ ]:
choices = {"auto": "auto", "required": "required", "none": "none",
           "named: get_weather": {"type": "function", "function": {"name": "get_weather"}}}
for question in ["Where is my order ORD55123?", "Tell me a joke about penguins."]:
    print(f"\n{question}")
    for label, tc_value in choices.items():
        r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}],
                                           tools=TOOLS, tool_choice=tc_value)
        calls = r.choices[0].message.tool_calls or []
        shown = ", ".join(f"{c.function.name}({c.function.arguments})" for c in calls) or "(text reply)"
        print(f"  {label:<20} -> {shown}")

| `tool_choice` | Behaviour | Use it for |
|---|---|---|
| `"auto"` (default) | the model decides: text or tool calls | normal agent turns |
| `"required"` | must call *some* tool, even for "tell me a joke" | the first step of a pipeline that must act |
| `"none"` | must answer in text; tools are visible but can't be called | the final "summarise what you found" turn |
| `{"type": "function", "function": {"name": ...}}` | must call *that* tool | forcing a classification or extraction tool, e.g. structured output via tools |

Look at the forced cases. `required` on the joke and `named: get_weather` on the order question produce calls whose arguments are made up (offline the stand-in fills in `London`), because you *forced* a tool that doesn't fit. Forcing is for when you already know which tool must run. (Ollama's OpenAI-compatible endpoint ignores `tool_choice`, so there everything behaves like `auto`.)

## 7. Tool errors are results, not exceptions

**In plain English:** when a tool fails, tell the *model*, in words it can act on. Crashing your loop helps nobody, and a raw stack trace wastes tokens and can leak internals.

In [ ]:
answer, _ = chat_with_tools("Where is my order ORD99999?")
assert "not found" in answer.lower()

A real model now says something like "I couldn't find order ORD99999. Could you double-check the number?" Good error messages are specific and suggest the fix ("status must be one of OPEN, CLOSED; you sent 'open'"). Also decide what's retryable: a timeout is worth one retry, a validation error with the same arguments is not. Cap failures per tool per run.

## 8. Validate arguments before executing

`strict: true` guarantees the *types*. It says nothing about business rules: that the order id is well-formed, that the refund is under the limit, that *this user* owns *this order*. Validate with pydantic between the proposal and the execution, and return the validation error as the tool result so the model can correct itself.

In [ ]:
class OrderStatusArgs(BaseModel):
    order_id: str

    @field_validator("order_id")
    @classmethod
    def well_formed(cls, v):
        if not re.fullmatch(r"ORD\d{5}", v):
            raise ValueError("order_id must be 'ORD' followed by 5 digits, e.g. ORD12345")
        return v

def guarded_order_status(raw_arguments, current_user_orders):
    try:
        args = OrderStatusArgs.model_validate_json(raw_arguments)       # 1) schema + format
    except ValidationError as e:
        return {"error": e.errors()[0]["msg"]}
    if args.order_id not in current_user_orders:                        # 2) authorisation, from YOUR data
        return {"error": "you can only look up your own orders"}
    return get_order_status(args.order_id)                               # 3) only now execute

mine = {"ORD55123"}
for raw in ['{"order_id": "ORD55123"}', '{"order_id": "55123"}', '{"order_id": "ORD70001"}']:
    print(f"{raw:<28} -> {guarded_order_status(raw, mine)}")
assert "error" in guarded_order_status('{"order_id": "55123"}', mine)

The checklist between "the model proposed a call" and "the call ran" (`po28`):

1. **Schema validation**: types, required fields, enums. Reject, don't coerce.
2. **Business limits in code**: a refund tool that accepts any amount will one day be asked for a very large one.
3. **Authorisation** against *your* systems, never inferred from the conversation ("the user said they're an admin").
4. **Rate and loop limits** per tool and per session.
5. **Human approval** for anything irreversible (payments, deletes, emails), with the proposed call shown in plain words (exercise 3).
6. **Audit log** of proposal, decision and result.

## 9. Never `eval` model output

The calculator is the classic temptation: the model sends `"17*23 + 4"`, and `eval()` would compute it. It would also run anything else a prompt injection talks the model into sending. The `safe_calc` above walks the syntax tree and allows arithmetic only:

In [ ]:
answer, _ = chat_with_tools("What is 17*23 + 4?")
print()
for expr in ["17*23 + 4", "2**10 - 1", "__import__('os').getcwd()", "open('secrets.txt').read()"]:
    print(f"{expr:<30} -> {calculator(expr)}")
assert calculator("17*23 + 4")["result"] == 395 and "error" in calculator("__import__('os').getcwd()")

The same rule covers SQL, shell and code tools: parameterised queries, allow-lists, sandboxes, never string-interpolated execution. Treat model output exactly like input from an anonymous user on the internet, because after one injected document, that's what it is.

## 10. The classic bugs: message ordering

The API checks the tool-call bookkeeping strictly. These two mistakes cause most 400s in hand-written agent loops:

In [ ]:
bug_1 = [{"role": "user", "content": "Where is my order ORD55123?"},
         {"role": "tool", "tool_call_id": "call_abc", "content": '{"status": "shipped"}'}]      # no tool call before it

call = {"id": "call_1", "type": "function", "function": {"name": "get_weather", "arguments": '{"city": "Paris", "unit": "celsius"}'}}
call_2 = {**call, "id": "call_2", "function": {**call["function"], "arguments": '{"city": "Tokyo", "unit": "celsius"}'}}
bug_2 = [{"role": "user", "content": "Weather in Paris and Tokyo?"},
         {"role": "assistant", "content": None, "tool_calls": [call, call_2]},
         {"role": "tool", "tool_call_id": "call_1", "content": '{"temp": 18}'},
         {"role": "user", "content": "And?"}]                                                    # call_2 never answered

for name, msgs in [("tool message without its tool call", bug_1), ("tool call left unanswered", bug_2)]:
    try:
        client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS)
        print(f"{name}: accepted by this server (OpenAI rejects it)")
    except openai.BadRequestError as e:
        print(f"{name}: 400 -> {api_message(e)[:150]}")

Causes you'll meet in real code: truncating history by message count, which cuts an assistant `tool_calls` message but keeps its `tool` replies (bug 1). Dropping or de-duplicating tool results (bug 2). Rebuilding the assistant message without its `tool_calls`. The fix: when you trim, trim **whole tool rounds**, i.e. the assistant message together with all its tool replies ([notebook 06](06_conversation_memory_and_context_windows.ipynb)).

## 11. Build the schema from the Python function

Hand-written schemas drift from the code they describe. Generate them from type hints and the docstring instead: pydantic does the JSON Schema, and a few lines make it strict.

In [ ]:
def get_weather_typed(city: Annotated[str, Field(description="City name, e.g. Paris")],
                      unit: Literal["celsius", "fahrenheit"] = "celsius") -> dict:
    """Get the current weather for one city. Call once per city."""
    return get_weather(city, unit)

def function_to_tool(fn):
    """Strict Chat Completions tool from a typed, documented function (flat arguments)."""
    hints = get_type_hints(fn, include_extras=True)
    fields = {name: (hints[name], ... if p.default is inspect.Parameter.empty else p.default)
              for name, p in inspect.signature(fn).parameters.items()}
    params = create_model(f"{fn.__name__}_args", **fields).model_json_schema()
    params.pop("title", None)
    for prop in params["properties"].values():
        prop.pop("title", None)
        if "default" in prop:              # strict mode: every key is required, so document the default instead
            prop["description"] = (prop.get("description", "") + f" (default: {prop.pop('default')})").strip()
    params.update(required=list(params["properties"]), additionalProperties=False)
    return {"type": "function", "function": {"name": fn.__name__, "description": inspect.getdoc(fn),
                                             "strict": True, "parameters": params}}

auto_tool = function_to_tool(get_weather_typed)
print(json.dumps(auto_tool, indent=1))
p = auto_tool["function"]["parameters"]
assert p["required"] == ["city", "unit"] and p["properties"]["unit"]["enum"] == ["celsius", "fahrenheit"]

r = client.chat.completions.create(model=MODEL, tools=[auto_tool],
                                   messages=[{"role": "user", "content": "What's the weather in Lima?"}])
tc = r.choices[0].message.tool_calls[0]
print("model called:", tc.function.name, tc.function.arguments)

For argument models you already have, `openai.pydantic_function_tool(MyArgs)` produces a strict tool definition in one line, and `client.chat.completions.parse(..., tools=[...])` then hands you `tool_call.function.parsed_arguments` as a typed object. The same idea is what MCP servers (FastMCP) and LangChain's `@tool` do: the schema comes from the signature, so it can't drift from the code ([MCP notebook](../15_agentic_ai/07_mcp_model_context_protocol.ipynb)).

## 12. The same loop in the Responses API

The Responses API uses a flatter tool format, typed output items, and `call_id` instead of `tool_call_id`:

In [ ]:
RESP_TOOLS = [{"type": "function", **WEATHER_TOOL["function"]}]          # flat: no nested "function" key
first = client.responses.create(model=MODEL, input="What's the weather in Oslo?", tools=RESP_TOOLS)
calls = [item for item in first.output if item.type == "function_call"]
for c in calls:
    print(f"function_call: call_id={c.call_id} name={c.name} arguments={c.arguments}")

outputs = [{"type": "function_call_output", "call_id": c.call_id,
            "output": json.dumps(REGISTRY[c.name](**json.loads(c.arguments)))} for c in calls]
final = client.responses.create(model=MODEL, previous_response_id=first.id, input=outputs, tools=RESP_TOOLS)
print("final:", final.output_text)
assert calls and "7" in final.output_text

| | Chat Completions | Responses API |
|---|---|---|
| Tool definition | `{"type": "function", "function": {"name", "description", "parameters", "strict"}}` | flat: `{"type": "function", "name", "description", "parameters", "strict"}` |
| The model's request | `message.tool_calls[i]`: `.id`, `.function.name`, `.function.arguments` | output item `type="function_call"`: `.call_id`, `.name`, `.arguments` |
| Returning a result | `{"role": "tool", "tool_call_id": id, "content": "..."}` after the assistant message | `{"type": "function_call_output", "call_id": id, "output": "..."}` |
| Conversation state | resend everything, incl. the assistant `tool_calls` message | `previous_response_id`, or resend the `function_call` items yourself |
| Built-in tools | none: every tool is your code | also hosted tools (web search, file search, code interpreter, remote MCP) |
| Portability | spoken by Ollama, vLLM, LiteLLM and most gateways | mainly OpenAI |

## Try it yourself

**1. Add a tool, and check the model picks it.** Write `convert_currency(amount, from_currency, to_currency)` with a fixed rate table (USD/EUR/GBP), build its schema with `function_to_tool` (use `Literal` for the currencies), and ask "Convert 250 USD to EUR". Check that it's the tool that gets called. (A real model fills `USD` → `EUR`. The offline stand-in fills both currency enums with the first currency it sees in the text, so there you'll see `USD` → `USD`, which is a reminder to validate arguments.)

In [ ]:
# Solution — try it yourself first, then run this
RATES = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27}
Currency = Literal["USD", "EUR", "GBP"]

def convert_currency(amount: float, from_currency: Currency, to_currency: Currency) -> dict:
    """Convert an amount of money between USD, EUR and GBP at today's fixed rate."""
    rate = 1.0 if from_currency == to_currency else RATES.get((from_currency, to_currency))
    if rate is None:
        return {"error": f"no rate for {from_currency}->{to_currency}"}
    return {"amount": round(amount * rate, 2), "currency": to_currency}

REGISTRY["convert_currency"] = convert_currency
tools_plus = TOOLS + [function_to_tool(convert_currency)]
r = client.chat.completions.create(model=MODEL, tools=tools_plus,
                                   messages=[{"role": "user", "content": "Convert 250 USD to EUR please, I need the money amount."}])
tc = r.choices[0].message.tool_calls[0]
print(tc.function.name, tc.function.arguments, "->", run_tools([tc])[0]["content"])
assert tc.function.name == "convert_currency"

**2. Guard against loops.** A buggy tool that always fails can make an agent retry forever. Add a *repeat detector* to the loop: if the model requests the same `(name, arguments)` pair a third time, stop and return a "giving up" message. Test it with a scripted sequence of identical calls. (No model is needed, since the detector is plain Python.)

In [ ]:
# Solution — try it yourself first, then run this
from collections import Counter

def should_stop(history_calls, new_calls, limit=3):
    """history_calls: Counter of (name, args) seen so far in this run."""
    for name, args in new_calls:
        history_calls[(name, args)] += 1
        if history_calls[(name, args)] >= limit:
            return f"giving up: {name}({args}) requested {limit} times"
    return None

seen = Counter()
proposals = [[("get_order_status", '{"order_id": "ORD00000"}')]] * 4          # the model keeps retrying
for step, calls in enumerate(proposals, 1):
    verdict = should_stop(seen, calls)
    print(f"step {step}: {verdict or 'run the calls'}")
    if verdict:
        break
assert step == 3

**3. Human approval for irreversible actions.** Add an `issue_refund(order_id, amount)` tool. In your executor, refuse to run it unless an `approve(call)` callback returns `True`, and refuse amounts over 100 outright. Simulate an approver that says yes only to amounts ≤ 50, and run three proposals through it.

In [ ]:
# Solution — try it yourself first, then run this
def issue_refund(order_id: str, amount: float) -> dict:
    """Refund an order. Irreversible: requires human approval."""
    return {"refunded": amount, "order_id": order_id}

def execute_with_approval(name, args, approve, hard_limit=100):
    if name == "issue_refund":
        if args["amount"] > hard_limit:
            return {"error": f"refunds over {hard_limit} are not allowed from chat"}      # limit in code, not in the prompt
        if not approve(f"Refund {args['amount']:.2f} on {args['order_id']}?"):
            return {"error": "a human declined this refund"}
    return {"issue_refund": issue_refund}[name](**args)

approver = lambda question: float(re.search(r"Refund ([\d.]+)", question).group(1)) <= 50
for amount in [30, 80, 500]:
    print(f"refund {amount:>3} -> {execute_with_approval('issue_refund', {'order_id': 'ORD55123', 'amount': amount}, approver)}")
assert "refunded" in execute_with_approval("issue_refund", {"order_id": "ORD55123", "amount": 30}, approver)

## Say it in an interview

- **The loop (30 s):** "I send messages plus tool schemas. If the reply has `tool_calls`, I append that assistant message, validate and execute each call, and append one `tool` message per call id. Then I call the model again, until it answers in text or I hit a step cap. The model never executes anything: it proposes, and my code decides."
- **Good tools:** verb_noun names; descriptions that say when *not* to use the tool (descriptions are prompts); enums for closed sets; `strict: true`; few tools per phase. Tool schemas are input tokens on every call, and a stable order keeps prompt caching working.
- **Between proposal and execution:** schema validation → business limits → authorisation against your own data → rate/loop limits → human approval for irreversible actions → audit log. Never `eval`.
- **Failures:** return errors *as tool results* in words the model can act on. Classify retryable vs not. Cap retries per tool and detect repeated identical calls.
- **Traps:** the `tool_call_id` bookkeeping (every call answered, no orphan tool messages); trimming history in whole tool rounds; forced `tool_choice` producing made-up arguments; `arguments` is a JSON *string*.
- **Function vs tool calling:** same mechanism, renamed. "Tool" also covers hosted tools, MCP servers and sub-agents.

## Next

- [Agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) · [Tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb) · [Guardrails, HITL and failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb)
- [MCP (Model Context Protocol)](../15_agentic_ai/07_mcp_model_context_protocol.ipynb) · [LangGraph ReAct tool-calling agent](../14_langgraph/05_react_tool_calling_agent.ipynb)
- [05 · Local LLMs with Ollama](05_local_llms_with_ollama.ipynb): the same tool loop against a local model
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)